# Full cMSCI Pipeline GPU Training

**Phase 2: AudioCaps Data Expansion + Ex-MCR + ProbVLM Training**

This notebook:
1. Downloads AudioCaps (~46K text-audio pairs)
2. Embeds captions with CLIP text encoder and audio with CLAP
3. Combines with existing 2,193 paired embeddings
4. Trains Ex-MCR projector (CLAP→CLIP) with InfoNCE
5. Retrains ProbVLM adapters on expanded data
6. Validates all models
7. ZIPs for download

**Upload:** `combined_training.npz` (existing 2K pairs) to the same folder.

**Expected time: ~20-40 minutes on T4/A100** (most time on AudioCaps embedding)

## Cell 1: GPU Check + Install Dependencies

In [ ]:
!pip install -q datasets transformers laion-clap soundfile librosa

import torch
import numpy as np
import json
import time
import os
from pathlib import Path

print("="*60)
print("GPU CHECK")
print("="*60)

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name}")
    print(f"  VRAM: {gpu_mem:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("  Using Apple MPS")
else:
    device = "cpu"
    print("  WARNING: No GPU found!")

print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {device}")
print("="*60)

## Cell 2: Download AudioCaps + Embed with CLIP/CLAP

AudioCaps provides ~46K (caption, audio_url) pairs. We embed:
- Captions with CLIP text encoder → 512-d CLIP text embeddings
- Captions with CLAP text encoder → 512-d CLAP text embeddings

Since CLIP text and CLIP image share the same space, CLIP text embeddings
serve as a proxy for image embeddings (conceptually: "what an image of this
caption would look like in CLIP space").

CLAP text embeddings provide the audio-space pairing.

This gives us paired (CLIP_text, CLAP_text) embeddings for Ex-MCR training.

In [ ]:
from datasets import load_dataset
from transformers import CLIPTokenizer, CLIPTextModel
import laion_clap

print("="*60)
print("STEP 1: Load AudioCaps Dataset")
print("="*60)

# Load AudioCaps from community upload
ds = load_dataset("d0rj/audiocaps")
print(f"  Train: {len(ds['train'])} samples")
print(f"  Val:   {len(ds['validation'])} samples")
print(f"  Test:  {len(ds['test'])} samples")

# Combine all splits
all_captions = []
for split in ['train', 'validation', 'test']:
    for item in ds[split]:
        caption = item.get('caption', item.get('text', ''))
        if caption and len(caption) > 5:
            all_captions.append(caption)

# Deduplicate
all_captions = list(set(all_captions))
print(f"  Unique captions: {len(all_captions)}")
print(f"  Example: '{all_captions[0]}'")

In [ ]:
print("="*60)
print("STEP 2: Embed Captions with CLIP Text Encoder")
print("="*60)

# Load CLIP text model
clip_model_name = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
clip_text_model = CLIPTextModel.from_pretrained(clip_model_name).to(device).eval()

BATCH_SIZE = 256
clip_embeddings = []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(all_captions), BATCH_SIZE):
        batch = all_captions[i:i+BATCH_SIZE]
        tokens = clip_tokenizer(batch, padding=True, truncation=True,
                                max_length=77, return_tensors="pt").to(device)
        outputs = clip_text_model(**tokens)
        # Use pooled output (CLS token)
        embs = outputs.pooler_output
        embs = embs / embs.norm(dim=-1, keepdim=True)  # L2 normalize
        clip_embeddings.append(embs.cpu().numpy())
        
        if (i // BATCH_SIZE + 1) % 20 == 0:
            print(f"  CLIP: {i+len(batch)}/{len(all_captions)} "
                  f"({100*(i+len(batch))/len(all_captions):.0f}%)")

clip_text_embs = np.concatenate(clip_embeddings, axis=0)
print(f"  CLIP text embeddings: {clip_text_embs.shape}")
print(f"  Completed in {time.time()-t0:.1f}s")

# Free GPU memory
del clip_text_model, clip_tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
print("="*60)
print("STEP 3: Embed Captions with CLAP Text Encoder")
print("="*60)

# Load CLAP model
clap_model = laion_clap.CLAP_Module(enable_fusion=False)
clap_model.load_ckpt()  # Downloads default checkpoint

BATCH_SIZE = 256
clap_embeddings = []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(all_captions), BATCH_SIZE):
        batch = all_captions[i:i+BATCH_SIZE]
        embs = clap_model.get_text_embedding(batch, use_tensor=False)
        # L2 normalize
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12
        embs = embs / norms
        clap_embeddings.append(embs)
        
        if (i // BATCH_SIZE + 1) % 20 == 0:
            print(f"  CLAP: {i+len(batch)}/{len(all_captions)} "
                  f"({100*(i+len(batch))/len(all_captions):.0f}%)")

clap_text_embs = np.concatenate(clap_embeddings, axis=0)
print(f"  CLAP text embeddings: {clap_text_embs.shape}")
print(f"  Completed in {time.time()-t0:.1f}s")

del clap_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Cell 3: Combine with Existing Data

In [ ]:
print("="*60)
print("STEP 4: Combine Training Data")
print("="*60)

# Load existing paired data
EXISTING_DATA_PATH = "combined_training.npz"
if not Path(EXISTING_DATA_PATH).exists():
    for alt in ["data/bridge_training/combined_training.npz",
                "../data/bridge_training/combined_training.npz"]:
        if Path(alt).exists():
            EXISTING_DATA_PATH = alt
            break

existing = np.load(EXISTING_DATA_PATH)
existing_clip = existing["image_embeddings"]  # CLIP image embeddings
existing_clap = existing["audio_embeddings"]  # CLAP audio embeddings
print(f"  Existing data: {len(existing_clip)} pairs")

# AudioCaps: CLIP text embeddings serve as CLIP-space vectors,
# CLAP text embeddings serve as CLAP-space vectors.
# Both describe the same concept, so they form valid pairs for
# learning the CLAP→CLIP projection.
print(f"  AudioCaps data: {len(clip_text_embs)} pairs")

# Combine
combined_clip = np.concatenate([existing_clip, clip_text_embs.astype(np.float32)], axis=0)
combined_clap = np.concatenate([existing_clap, clap_text_embs.astype(np.float32)], axis=0)

print(f"  Combined: {len(combined_clip)} total pairs")
print(f"    CLIP shape: {combined_clip.shape}")
print(f"    CLAP shape: {combined_clap.shape}")

# Save combined data
np.savez_compressed(
    "combined_training_v2.npz",
    image_embeddings=combined_clip,  # Keep key name for compatibility
    audio_embeddings=combined_clap,
)
v2_size = Path("combined_training_v2.npz").stat().st_size / (1024*1024)
print(f"  Saved: combined_training_v2.npz ({v2_size:.1f} MB)")

## Cell 4: Train Ex-MCR Projector (CLAP → CLIP)

The key model: an MLP that projects CLAP audio embeddings into CLIP space.
After training, `ExMCR(audio_clap)` can be compared directly to CLIP text/image.

Architecture: 512 → 512 → 512 (ReLU, L2-norm)
Loss: Symmetric InfoNCE with learnable temperature

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

# ─── Ex-MCR Model ───────────────────────────────────────────

class ExMCRNet(nn.Module):
    """MLP projector: CLAP 512-d → CLIP 512-d with L2 normalization."""
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=512):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.layers(x), p=2, dim=-1)


class InfoNCELoss(nn.Module):
    """Symmetric InfoNCE with learnable temperature."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.tensor(np.log(1.0 / temperature)))
    @property
    def temperature(self):
        return torch.exp(-self.log_temperature)
    def forward(self, anchor, positive):
        logits = torch.mm(anchor, positive.t()) / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))
        with torch.no_grad():
            acc = ((logits.argmax(1) == labels).float().mean() +
                   (logits.t().argmax(1) == labels).float().mean()) / 2
        return loss, {"loss": loss.item(), "acc": acc.item(), "temp": self.temperature.item()}


class PairedDataset(Dataset):
    def __init__(self, clip_embs, clap_embs):
        self.clip = torch.tensor(clip_embs, dtype=torch.float32)
        self.clap = torch.tensor(clap_embs, dtype=torch.float32)
    def __len__(self): return len(self.clip)
    def __getitem__(self, i): return self.clip[i], self.clap[i]


# ─── Training ────────────────────────────────────────────────

print("="*60)
print("TRAINING: Ex-MCR Projector (CLAP → CLIP)")
print("="*60)

EXMCR_EPOCHS = 50
EXMCR_BATCH_SIZE = 256  # Larger batch for contrastive learning
EXMCR_LR = 3e-4
EXMCR_PATIENCE = 10

EXMCR_DIR = Path("trained_models/exmcr")
EXMCR_DIR.mkdir(parents=True, exist_ok=True)

# L2-normalize CLIP targets
clip_norms = np.linalg.norm(combined_clip, axis=1, keepdims=True) + 1e-12
clip_normed = combined_clip / clip_norms

dataset = PairedDataset(clip_normed, combined_clap)
n_val = max(1, int(len(dataset) * 0.15))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=EXMCR_BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=EXMCR_BATCH_SIZE, shuffle=False)

model = ExMCRNet().to(device)
loss_fn = InfoNCELoss().to(device)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=EXMCR_LR, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXMCR_EPOCHS)

n_params = sum(p.numel() for p in model.parameters())
print(f"  Train: {n_train}, Val: {n_val}, Batch: {EXMCR_BATCH_SIZE}")
print(f"  Parameters: {n_params:,}")

best_val_loss = float("inf")
patience_counter = 0
t_start = time.time()

for epoch in range(EXMCR_EPOCHS):
    model.train(); loss_fn.train()
    train_metrics = []
    for clip_b, clap_b in train_loader:
        clip_b, clap_b = clip_b.to(device), clap_b.to(device)
        optimizer.zero_grad()
        projected = model(clap_b)
        anchor = F.normalize(clip_b, p=2, dim=-1)
        loss, metrics = loss_fn(anchor, projected)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_metrics.append(metrics)
    scheduler.step()
    
    avg_train = {k: np.mean([m[k] for m in train_metrics]) for k in train_metrics[0]}
    
    model.eval(); loss_fn.eval()
    val_losses = []
    with torch.no_grad():
        for clip_b, clap_b in val_loader:
            clip_b, clap_b = clip_b.to(device), clap_b.to(device)
            projected = model(clap_b)
            anchor = F.normalize(clip_b, p=2, dim=-1)
            loss, _ = loss_fn(anchor, projected)
            val_losses.append(loss.item())
    val_loss = np.mean(val_losses)
    
    print(f"  Epoch {epoch+1:3d}/{EXMCR_EPOCHS}: loss={avg_train['loss']:.4f} "
          f"acc={avg_train['acc']:.3f} val_loss={val_loss:.4f} temp={avg_train['temp']:.3f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.layers.state_dict(), EXMCR_DIR / "ex_clap.pt")
    else:
        patience_counter += 1
        if patience_counter >= EXMCR_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

elapsed = time.time() - t_start
print(f"\n  Ex-MCR training complete in {elapsed:.1f}s")
print(f"  Best val_loss: {best_val_loss:.4f}")
print(f"  Saved: {EXMCR_DIR / 'ex_clap.pt'}")

## Cell 5: Retrain Probabilistic Adapters on Expanded Data

In [ ]:
# ─── ProbVLM Adapter ────────────────────────────────────────

class ProbabilisticAdapter(nn.Module):
    """BayesCap/ProbVLM: predicts (mu, alpha, beta) for Generalized Gaussian."""
    def __init__(self, input_dim=512, hidden_dim=256, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        layers = []
        in_d = input_dim
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(in_d, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            in_d = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(hidden_dim, input_dim)
        self.alpha_head = nn.Linear(hidden_dim, input_dim)
        self.beta_head = nn.Linear(hidden_dim, input_dim)
        self.config = dict(input_dim=input_dim, hidden_dim=hidden_dim,
                           num_layers=num_layers, dropout=dropout)

    def forward(self, embedding):
        h = self.backbone(embedding)
        mu = embedding + self.mu_head(h)
        alpha = F.softplus(self.alpha_head(h)).clamp(1e-6, 50.0)
        beta = F.softplus(self.beta_head(h)).clamp(0.5, 10.0)
        return mu, alpha, beta

    def uncertainty(self, emb_np):
        self.eval()
        emb = emb_np.squeeze()
        if emb.ndim == 1: emb = emb[np.newaxis, :]
        with torch.no_grad():
            _, alpha, _ = self.forward(torch.tensor(emb, dtype=torch.float32))
            return float(alpha.mean().item())

    def save(self, path):
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)
        with p.with_suffix('.json').open('w') as f: json.dump(self.config, f, indent=2)

    @classmethod
    def load(cls, path):
        p = Path(path)
        with p.with_suffix('.json').open('r') as f: config = json.load(f)
        m = cls(**config)
        m.load_state_dict(torch.load(p, map_location='cpu', weights_only=True))
        return m.eval()


class GenGaussNLL(nn.Module):
    def forward(self, mu, alpha, beta, target):
        residual = torch.abs(target - mu) + 1e-8
        alpha_c = torch.clamp(alpha, min=1e-4)
        beta_c = torch.clamp(beta, min=0.5, max=10.0)
        # Clamp the power term to prevent inf/nan
        power = torch.clamp((residual / alpha_c).pow(beta_c), max=100.0)
        return (torch.log(alpha_c) + power).mean()


class EmbPairDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.tensor(inputs, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
    def __len__(self): return len(self.inputs)
    def __getitem__(self, i): return self.inputs[i], self.targets[i]


def train_prob_adapter(embeddings, name, output_path, epochs=100, patience=15):
    print(f"\n{'='*60}")
    print(f"TRAINING: {name} Probabilistic Adapter")
    print(f"{'='*60}")

    rng = np.random.default_rng(42)
    noise = rng.normal(0, 0.01, size=embeddings.shape).astype(np.float32)
    targets = embeddings + noise

    dataset = EmbPairDataset(embeddings, targets)
    n_val = max(1, int(len(dataset) * 0.15))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                              drop_last=len(train_ds) > 64)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    adapter = ProbabilisticAdapter(input_dim=512).to(device)
    opt = torch.optim.AdamW(adapter.parameters(), lr=5e-5, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    l1 = nn.L1Loss()
    gg = GenGaussNLL()

    n_params = sum(p.numel() for p in adapter.parameters())
    print(f"  Train: {n_train}, Val: {n_val}, Params: {n_params:,}")

    best_val = float("inf")
    pat = 0
    t0 = time.time()

    for ep in range(epochs):
        adapter.train()
        losses = []
        for inp, tgt in train_loader:
            inp, tgt = inp.to(device), tgt.to(device)
            opt.zero_grad()
            mu, alpha, beta = adapter(inp)
            loss = l1(mu, tgt) + 0.1 * gg(mu, alpha, beta, tgt)
            if torch.isnan(loss):
                continue  # Skip nan batches
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sched.step()

        if not losses:
            print(f"  Epoch {ep+1}: all batches nan, stopping")
            break

        adapter.eval()
        vlosses = []
        with torch.no_grad():
            for inp, tgt in val_loader:
                inp, tgt = inp.to(device), tgt.to(device)
                mu, alpha, beta = adapter(inp)
                vl = l1(mu, tgt) + 0.1 * gg(mu, alpha, beta, tgt)
                if not torch.isnan(vl):
                    vlosses.append(vl.item())

        vl = np.mean(vlosses) if vlosses else float("inf")

        if (ep+1) % 10 == 0 or ep == 0:
            print(f"  Epoch {ep+1:3d}/{epochs}: train={np.mean(losses):.4f} val={vl:.4f}")

        if vl < best_val:
            best_val = vl; pat = 0
            adapter.save(output_path)
        else:
            pat += 1
            if pat >= patience:
                print(f"  Early stopping at epoch {ep+1}")
                break

    print(f"  Done in {time.time()-t0:.1f}s (best val={best_val:.4f})")
    return ProbabilisticAdapter.load(output_path).to(device)


ADAPTER_DIR = Path("trained_models/prob_adapters")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# Train CLIP adapter on CLIP-space embeddings (image + text)
clip_adapter = train_prob_adapter(
    combined_clip, "CLIP", str(ADAPTER_DIR / "clip_adapter.pt")
)

# Train CLAP adapter on CLAP-space embeddings
clap_adapter = train_prob_adapter(
    combined_clap, "CLAP", str(ADAPTER_DIR / "clap_adapter.pt")
)

## Cell 6: Validate All Models

In [ ]:
print("="*60)
print("VALIDATION: All Models")
print("="*60)

# ─── Ex-MCR Validation ──────────────────────────────────────
print("\n--- Ex-MCR Projector ---")

# Reload best
exmcr_model = ExMCRNet().to(device)
state = torch.load(EXMCR_DIR / "ex_clap.pt", map_location=device, weights_only=True)
exmcr_model.layers.load_state_dict(state)
exmcr_model.eval()

# Test on validation set
n_test = min(500, n_val)
rng = np.random.default_rng(123)
test_idx = rng.choice(len(combined_clip), n_test, replace=False)

with torch.no_grad():
    clip_t = torch.tensor(clip_normed[test_idx], dtype=torch.float32).to(device)
    clap_t = torch.tensor(combined_clap[test_idx], dtype=torch.float32).to(device)
    projected = exmcr_model(clap_t)
    anchor = F.normalize(clip_t, p=2, dim=-1)
    
    # Matched similarities (diagonal)
    matched = (anchor * projected).sum(dim=1)
    
    # Mismatched (off-diagonal mean)
    sim_matrix = torch.mm(anchor, projected.t())
    mask = ~torch.eye(n_test, dtype=torch.bool, device=device)
    mismatched = sim_matrix[mask]

print(f"  Matched similarity:    {matched.mean().item():.4f} +/- {matched.std().item():.4f}")
print(f"  Mismatched similarity: {mismatched.mean().item():.4f} +/- {mismatched.std().item():.4f}")
print(f"  Gap:                   {matched.mean().item() - mismatched.mean().item():.4f}")
print(f"  Result: {'PASS' if matched.mean() > mismatched.mean() else 'FAIL'}")

# ─── ProbVLM Validation ─────────────────────────────────────
print("\n--- Probabilistic Adapters ---")
n_check = 200

clip_uncs = [clip_adapter.uncertainty(combined_clip[i]) for i in range(n_check)]
clap_uncs = [clap_adapter.uncertainty(combined_clap[i]) for i in range(n_check)]

print(f"  CLIP uncertainty: mean={np.mean(clip_uncs):.6f}, std={np.std(clip_uncs):.6f}")
print(f"  CLAP uncertainty: mean={np.mean(clap_uncs):.6f}, std={np.std(clap_uncs):.6f}")
clip_cv = np.std(clip_uncs) / (np.mean(clip_uncs) + 1e-10)
clap_cv = np.std(clap_uncs) / (np.mean(clap_uncs) + 1e-10)
print(f"  CLIP CV: {clip_cv:.4f}, CLAP CV: {clap_cv:.4f}")
print(f"  Result: {'PASS' if clip_cv > 0.01 and clap_cv > 0.01 else 'LOW VARIATION'}")

## Cell 7: ZIP + Download

In [ ]:
import shutil

print("="*60)
print("ALL TRAINING COMPLETE!")
print("="*60)
print()
print("Trained models:")
for p in sorted(Path("trained_models").rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p}: {size_kb:.1f} KB")

# Also include the expanded training data
shutil.copy("combined_training_v2.npz", "trained_models/combined_training_v2.npz")

zip_path = shutil.make_archive("trained_models_full_pipeline", "zip", ".", "trained_models")
zip_size = Path(zip_path).stat().st_size / (1024*1024)
print(f"\nCreated: {zip_path} ({zip_size:.1f} MB)")
print()
print("="*60)
print("DEPLOYMENT INSTRUCTIONS")
print("="*60)
print()
print("1. Download trained_models_full_pipeline.zip")
print("2. Unzip and copy files:")
print("     exmcr/ex_clap.pt → models/exmcr/ex_clap.pt")
print("     bridge/bridge_best.pt → models/bridge/bridge_best.pt")
print("     bridge/bridge_best.json → models/bridge/bridge_best.json")
print("     prob_adapters/clip_adapter.pt → models/prob_adapters/clip_adapter.pt")
print("     prob_adapters/clip_adapter.json → models/prob_adapters/clip_adapter.json")
print("     prob_adapters/clap_adapter.pt → models/prob_adapters/clap_adapter.pt")
print("     prob_adapters/clap_adapter.json → models/prob_adapters/clap_adapter.json")
print("     combined_training_v2.npz → data/bridge_training/combined_training_v2.npz")
print()
print("3. On your laptop, run the full pipeline:")
print("     python scripts/extend_calibration.py")
print("     python scripts/optimize_cmsci.py")
print("     python scripts/analyze_rq3.py")
print("     python scripts/run_cmsci_comparison.py --all")
print("     python scripts/run_cmsci_ablation.py")
print("     python scripts/generate_paper_figures.py")